In [4]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from embeddings import compute_EIF_a0, compute_plugin_a0, MMD, MMD_plugin, MMD_dr
from DGPs import DGP_univariate, DGP_high_dim_X, DGP_high_dim_M, DGP_high_dim_MX, DGP_high_dim_Y, DGP_binary
from KRR_utils import ard_median_heuristic, learn_hyperparams

In [5]:
n_list=[250,500,1000,2000,4000,8000]
treatment=1
method_list=['logistic','histgbm','reggbm','forest']
num_seeds=50


In [ ]:
def single_loop(DGP,Y_truth,seed,n,dim,a0,confounding,x_method,mx_method,params,explicit_m):
    A,M,X,Y = DGP(2*n,seed=seed,dim=dim,confounding=confounding)
    A_tr=A[:n,:]
    A_te=A[n:,:]
    M_tr=M[:n,:]
    M_te=M[n:,:]
    X_tr=X[:n,:]
    X_te=X[n:,:]
    Y_tr=Y[:n,:]
    Y_te=Y[n:,:]
    dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=a0,x_method=x_method,mx_method=mx_method,params=params,explicit_m=explicit_m)
    div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
    plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=a0,x_method=x_method,params=params,explicit_m=explicit_m)
    div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
    return div_dr,div_pi

In [3]:
#binary DGP

dim=3

A_truth,M_truth,X_truth,Y_truth = DGP_binary(10000,seed=101,dim=dim,gt=True,a0=treatment,confounding=True)

A,M,X,Y = DGP_binary(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

KeyboardInterrupt: 

In [3]:
dr_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))

for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_binary(2*n,seed=k,dim=dim,confounding=True)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)

with open('binary_confounded.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

n:  250
0
25
0
25
0
25
0
25
n:  500
0
25
0
25
0
25
0
25
n:  1000
0
25
0
25
0
25
0
25
n:  2000
0
25
0
25
0
25
0
25
n:  4000
0
25
0
25
0
25
0
25
n:  8000
0
25
0
25
0
25
0
25


In [6]:
dim=3

A_truth,M_truth,X_truth,Y_truth = DGP_binary(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_binary(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

In [9]:
#binary DGP



dr_mat=jnp.zeros((len(n_list),2,num_seeds))
pi_mat=jnp.zeros((len(n_list),2,num_seeds))
method='table'
truth_table=[True,False]
for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(2):
        explicit_m=truth_table[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_binary(2*n,seed=k,dim=dim)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params,explicit_m=explicit_m)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            #print(div_dr)
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params,explicit_m=explicit_m)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)

with open('binary_tabular.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

n:  250
0
25
0
25
n:  500
0
25
0
25
n:  1000
0
25
0
25
n:  2000
0
25
0
25
n:  4000
0
25
0
25
n:  8000
0
25
0
25


KeyboardInterrupt: 

In [10]:
with open('binary_tabular.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

In [ ]:
#univariate DGP

dim=3

A_truth,M_truth,X_truth,Y_truth = DGP_univariate(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_univariate(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

dr_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))

for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_univariate(2*n,seed=k,dim=dim)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)

with open('univariate.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

In [ ]:
#high dimensional X DGP

dim=50

A_truth,M_truth,X_truth,Y_truth = DGP_high_dim_X(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_high_dim_X(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

dr_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))

for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_high_dim_X(2*n,seed=k,dim=dim)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)


with open('high_dim_X.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

In [ ]:
#high dimensional M and X DGP

dim=20

A_truth,M_truth,X_truth,Y_truth = DGP_high_dim_MX(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_high_dim_MX(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

dr_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))

for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_high_dim_MX(2*n,seed=k,dim=dim)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)

with open('high_dim_MX.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

In [2]:
n_list=[250,500,1000,2000,4000,8000]
treatment=1
method_list=['forest']
num_seeds=50


In [3]:
#univariate DGP

dim=3

A_truth,M_truth,X_truth,Y_truth = DGP_univariate(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_univariate(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

dr_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))

for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_univariate(2*n,seed=k,dim=dim)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)

with open('univariate_forest.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

n:  250
0
25
n:  500
0
25
n:  1000
0
25
n:  2000
0
25
n:  4000
0
25
n:  8000
0
25


In [4]:
#high dimensional X DGP

dim=50

A_truth,M_truth,X_truth,Y_truth = DGP_high_dim_X(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_high_dim_X(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

dr_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))

for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_high_dim_X(2*n,seed=k,dim=dim)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)


with open('high_dim_X_forest.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

n:  250
0
25
n:  500
0
25
n:  1000
0
25
n:  2000
0
25
n:  4000
0
25
n:  8000
0
25


In [5]:
#high dimensional M and X DGP

dim=20

A_truth,M_truth,X_truth,Y_truth = DGP_high_dim_MX(10000,seed=101,dim=dim,gt=True,a0=treatment)

A,M,X,Y = DGP_high_dim_MX(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

dr_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(n_list),len(method_list),num_seeds))

for i in range(len(n_list)):
    n=n_list[i]
    print('n: ',n)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_high_dim_MX(2*n,seed=k,dim=dim)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)

with open('high_dim_MX_forest.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

n:  250
0
25
n:  500
0
25
n:  1000
0
25
n:  2000
0
25
n:  4000
0
25
n:  8000
0
25


In [4]:
treatment=1
method_list=['logistic','histgbm','reggbm','forest']
num_seeds=50
n=1000
u_list=[0.1,0.5,1,5,10]
dim=3

A,M,X,Y = DGP_univariate(2000,seed=0,dim=dim)

params=learn_hyperparams(A,M,X,Y,subsample=False)

dr_mat=jnp.zeros((len(u_list),len(method_list),num_seeds))
pi_mat=jnp.zeros((len(u_list),len(method_list),num_seeds))

for i in range(len(u_list)):
    u=u_list[i]
    print('u: ',u)
    A_truth,M_truth,X_truth,Y_truth = DGP_univariate(10000,seed=101,dim=dim,gt=True,a0=treatment,u_scale=u)
    for j in range(len(method_list)):
        method=method_list[j]
        for k in range(num_seeds):
            if k%25 == 0:
                print(k)
            A,M,X,Y = DGP_univariate(2*n,seed=k,dim=dim,u_scale=u)
            A_tr=A[:n,:]
            A_te=A[n:,:]
            M_tr=M[:n,:]
            M_te=M[n:,:]
            X_tr=X[:n,:]
            X_te=X[n:,:]
            Y_tr=Y[:n,:]
            Y_te=Y[n:,:]
        
            dr,EIF = compute_EIF_a0(A_tr,M_tr,X_tr,Y_tr,A_te,M_te,X_te,Y_te,a0=treatment,x_method=method,mx_method=method,params=params)
            div_dr = jnp.sqrt(MMD_dr(Y_te,Y_tr,Y_truth,EIF,dr))
            dr_mat = dr_mat.at[i,j,k].set(div_dr)

            plugin = compute_plugin_a0(A_tr,M_tr,X_tr,Y_tr,a0=treatment,x_method=method,params=params)
            div_pi = jnp.sqrt(MMD_plugin(Y_tr,Y_truth,plugin))
            pi_mat = pi_mat.at[i,j,k].set(div_pi)

with open('univariate_confounded.npy', 'wb') as f:
    jnp.save(f, dr_mat)
    jnp.save(f, pi_mat)

u:  0.1
0
25
0
25
0
25
0
25
u:  0.5
0
25
0
25
0
25
0
25
u:  1
0
25
0
25
0
25
0
25
u:  5
0
25
0
25
0
25
0
25
u:  10
0
25
0
25
0
25
0
25
